# Dependencies

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib
import os

# understand what our CSV

In [ ]:
import os
import pandas as pd

# 1. Search across local directory, parent directory, and Kaggle input folders
candidate_files = []

# Scan /kaggle/input if on Kaggle
if os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        for file in files:
            if file.endswith('.csv'):
                candidate_files.append(os.path.join(root, file))

# Add standard local paths
candidate_files.extend([
    'Crop_recommendation.csv',
    '../data/Crop_recommendation.csv',
    '../../data/Crop_recommendation.csv',
    'data/Crop_recommendation.csv',
    'ml_model/data/Crop_recommendation.csv'
])

# Find first existing path containing 'crop'
data_path = next((p for p in candidate_files if os.path.exists(p) and 'crop' in p.lower()), None)

# If still not found, check any available CSV file
if data_path is None:
    data_path = next((p for p in candidate_files if os.path.exists(p)), None)

if data_path:
    print(f"✔ Successfully located dataset at: {data_path}")
    df = pd.read_csv(data_path)
else:
    # Try internet download if Internet is enabled in Kaggle Settings
    github_url = 'https://raw.githubusercontent.com/Sonu0Sharma/CropGuru___ML_based_crop_recommendation_system/main/ml_model/data/Crop_recommendation.csv'
    try:
        print("Searching via online repository...")
        df = pd.read_csv(github_url)
        print("✔ Loaded directly from GitHub!")
    except Exception:
        raise FileNotFoundError(
            "Dataset not found! On Kaggle, please click '+ Add Input' in the right sidebar "
            "and attach 'Crop_recommendation.csv', or toggle 'Internet: On' in Notebook Options."
        )

df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

# Performing EDA ~

In [ ]:
# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set a style for the plots for better aesthetics
plt.style.use('seaborn-v0_8-darkgrid') # Using seaborn darkgrid for a clean look
# You can also try 'ggplot', 'fivethirtyeight', 'bmh'

In [ ]:
print("Starting Exploratory Data Analysis (EDA) on the Crop Recommendation Dataset... 📈\n")


In [ ]:
try:
    df = pd.read_csv(data_path)
    print(f"Dataset loaded successfully from: {data_path} ✅")
except FileNotFoundError:
    print(f"Error: '{data_path}' not found. Please ensure the CSV file is in the correct directory. ❌")
    print("Expected path (relative to your notebook): CropGuru/ml_model/data/Crop_recommendation.csv")
    exit() # Exit the script if the file isn't found

In [ ]:
# --- 2. Initial Data Inspection (as you previously ran) ---
print("\n--- Initial Dataframe Information ---")
df.info()
print("\nThis provides a quick overview of column names, non-null counts, and data types.")


In [ ]:
print("\n--- First 5 Rows of the Dataset ---")
print(df.head())
print("\nViewing the top rows helps to quickly understand the data's structure and content.")



In [ ]:
print("\n--- Summary Statistics of Numerical Columns ---")
print(df.describe())
print("\nDescriptive statistics give insights into the central tendency, dispersion, and shape of numerical features.")


In [ ]:
print("\n--- Unique Crop Types (Labels) ---")
unique_crops = df['label'].unique()
print(unique_crops)
print(f"\nThere are {len(unique_crops)} unique crop types in the dataset.")


In [ ]:
# --- 3. Univariate Analysis: Distribution of Crop Types ---
print("\n--- Plotting: Distribution of Crop Types ---")
plt.figure(figsize=(15, 8))
# Count plot to show the frequency of each crop type
sns.countplot(y='label', data=df, order=df['label'].value_counts().index, palette='viridis')
plt.title('Distribution of Crop Types', fontsize=18, fontweight='bold')
plt.xlabel('Count', fontsize=14)
plt.ylabel('Crop Type', fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
print("Insight: This chart clearly shows that each of the 22 crop types is equally represented with 100 samples each. This is a well-balanced dataset for classification tasks. 📊")


In [ ]:
# --- 4. Univariate Analysis: Distribution of Numerical Features ---
print("\n--- Plotting: Distribution of Numerical Features (Histograms) ---")
numerical_features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
# Create histograms for each numerical feature to show their distributions
df[numerical_features].hist(bins=30, figsize=(20, 15), edgecolor='black', color='skyblue', alpha=0.7)
plt.suptitle('Distribution of Numerical Features', y=1.02, fontsize=20, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.98]) # Adjust layout to prevent suptitle overlap
plt.show()
print("Insight: These histograms display the frequency distribution of each environmental and nutrient factor. They help in understanding the range and common values for each feature, identifying any skewed distributions or potential outliers. 📈")


In [ ]:
# --- 5. Bivariate Analysis: Correlation Heatmap ---
print("\n--- Plotting: Correlation Matrix of Numerical Features ---")
plt.figure(figsize=(10, 8))
# Calculate the correlation matrix
correlation_matrix = df[numerical_features].corr()
# Create a heatmap to visualize the correlations between numerical features
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5, cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Correlation Matrix of Numerical Features', fontsize=18, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()
print("Insight: The heatmap reveals the linear relationships between the numerical variables. A notable positive correlation is observed between 'P' (Phosphorus) and 'K' (Potassium), suggesting that these two nutrients often co-vary. 🔗")


In [ ]:
# --- 6. Bivariate Analysis: Box Plots showing Feature Distribution by Crop Type ---
print("\n--- Plotting: Feature Distribution Across Different Crop Types (Box Plots) ---")
# Generate box plots for each numerical feature, showing its distribution across different crop types
for feature in numerical_features:
    plt.figure(figsize=(15, 8))
    sns.boxplot(x='label', y=feature, data=df, palette='Spectral')
    plt.title(f'{feature} Distribution Across Different Crop Types', fontsize=18, fontweight='bold')
    plt.xlabel('Crop Type', fontsize=14)
    plt.ylabel(feature, fontsize=14)
    plt.xticks(rotation=45, ha='right', fontsize=10) # Rotate labels for better readability
    plt.yticks(fontsize=10)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
    print(f"Insight for {feature}: These box plots are crucial for understanding the ideal range of values for '{feature}' required by each specific crop. They visually highlight the distinct environmental and nutrient preferences of different crops. 🌱")

print("\nComprehensive EDA complete! These visualizations provide deep insights into the dataset, which is invaluable for building a robust crop recommendation model. 🎉")

## since data set in already healthy -- we will proceed to Prepare the data for training

In [ ]:
# The features (inputs) are the soil and environmental data
X = df[['N', 'P', 'K', 'ph', 'temperature', 'humidity', 'rainfall']]

# The target (what we want to predict) is the crop label
y = df['label']

# Split the data into a "Training Set" and a "Testing Set"

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# Create and Train the Model

In [ ]:
# Create the model instance
model = RandomForestClassifier(n_estimators=100, random_state=42)

# "Fit" the model to our training data
model.fit(X_train, y_train)

print("Model training complete!")

# Performance  Check 

In [ ]:
# Make predictions on the test data
y_pred = model.predict(X_test)

# Calculate the accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy * 100:.2f}%")

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# If not, please run your previous model training code first.
# X_test, X_train, y_test, y_train, y_pred, df, etc. should already exist.

# Get the unique crop labels for consistent ordering
labels = sorted(df['label'].unique())

# --- Confusion Matrix Plot ---
print("Generating Confusion Matrix Plot...")
plt.figure(figsize=(16, 14))
cm = confusion_matrix(y_test, y_pred, labels=labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels,
            linewidths=.5, linecolor='lightgray')
plt.title('Confusion Matrix: Actual vs. Predicted Crop Labels', fontsize=22, fontweight='bold', pad=20)
plt.xlabel('Predicted Label', fontsize=16, labelpad=15)
plt.ylabel('True Label', fontsize=16, labelpad=15)
plt.xticks(rotation=90, fontsize=12)
plt.yticks(rotation=0, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# --- Classification Report Metrics as Graphs ---
print("\nGenerating Precision, Recall, and F1-Score Plots...")
report = classification_report(y_test, y_pred, target_names=labels, output_dict=True)


In [ ]:
# Convert the report to a DataFrame for easier plotting
metrics_df = pd.DataFrame(report).transpose().drop(['accuracy', 'macro avg', 'weighted avg'])
metrics_df['Class'] = metrics_df.index
metrics_melted = metrics_df.melt(id_vars='Class', value_vars=['precision', 'recall', 'f1-score'],
                                 var_name='Metric', value_name='Score')

# Sort classes by F1-score for consistent plotting order
metrics_melted_sorted = metrics_melted[metrics_melted['Metric'] == 'f1-score'].sort_values(by='Score', ascending=False)
sorted_classes = metrics_melted_sorted['Class'].tolist()

# Plotting individual metrics
for metric in ['precision', 'recall', 'f1-score']:
    plt.figure(figsize=(16, 8))
    sns.barplot(x='Class', y='Score', data=metrics_melted[metrics_melted['Metric'] == metric],
                order=sorted_classes, palette='viridis')
    plt.title(f'{metric.capitalize()} per Crop Type', fontsize=20, fontweight='bold', pad=15)
    plt.xlabel('Crop Type', fontsize=14)
    plt.ylabel(f'{metric.capitalize()} Score', fontsize=14)
    plt.xticks(rotation=45, ha='right', fontsize=12)
    plt.yticks(fontsize=12)
    plt.ylim(0.0, 1.05)
    plt.tight_layout()
    plt.show()
    print(f"{metric.capitalize()} plot generated. ✅\n")

In [ ]:
# --- Optional: Combined Metrics Plot ---
print("Generating Combined Precision, Recall, and F1-Score Plot...")
plt.figure(figsize=(18, 10))
sns.barplot(x='Class', y='Score', hue='Metric', data=metrics_melted,
            order=sorted_classes, palette='dark')
plt.title('Precision, Recall, and F1-Score for Each Crop Type', fontsize=22, fontweight='bold', pad=20)
plt.xlabel('Crop Type', fontsize=16, labelpad=15)
plt.ylabel('Score', fontsize=16, labelpad=15)
plt.xticks(rotation=60, ha='right', fontsize=12)
plt.yticks(fontsize=12)
plt.ylim(0.0, 1.05)
plt.legend(title='Metric', title_fontsize='13', fontsize='12', loc='lower right')
plt.tight_layout()
plt.show()
print("Combined plot generated. ✅\n")

print("All performance visualizations generated successfully! ✨")


# Save the trained model to a file

In [ ]:
# Construct the path to save the model in the 'src' directory
model_path = os.path.join('..', 'src', 'crop_recommendation_model.pkl')

# Save the model object
joblib.dump(model, model_path)

print(f"Model saved to {model_path}")